# P1 — Revenue Forecasting Demo

Load **LightGBM** và **LSTM** từ MLflow Registry, backtesting trên 20% Val set, vẽ biểu đồ so sánh.


In [1]:
import os, sys, math
import numpy as np
import pandas as pd
import torch
import mlflow, mlflow.lightgbm, mlflow.pytorch, mlflow.sklearn
import plotly.graph_objects as go
from mlflow.tracking import MlflowClient
from dotenv import load_dotenv
from sklearn.metrics import mean_squared_error, mean_absolute_error


sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

os.environ['MLFLOW_S3_ENDPOINT_URL']  = f"http://{os.getenv('MINIO_ENDPOINT')}"
os.environ['AWS_ACCESS_KEY_ID']       = os.getenv('MINIO_ACCESS_KEY')
os.environ['AWS_SECRET_ACCESS_KEY']   = os.getenv('MINIO_SECRET_KEY')
os.environ['MLFLOW_S3_IGNORE_TLS']    = 'true'
mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI'))

print('✅ Setup xong.')

c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup xong.


## 1. Import hàm từ train.py

In [2]:
from src.mlops.data_loader import load_revenue_data
from src.mlops.p1_revenue.train import (
    build_lag_features,
    build_sequences,
    FEATURE_COLS,
    CALENDAR_COLS,
    TARGET_COL,
    TRAIN_RATIO,
    SEQ_LEN,
)

LGB_FEATURES = (
    CALENDAR_COLS
    + [f'revenue_lag_{i}'            for i in [1, 2, 3, 4, 8]]
    + [f'order_count_lag_{i}'        for i in [1, 2, 3, 4, 8]]
    + [f'weekend_order_ratio_lag_{i}' for i in [1, 2, 3, 4, 8]]
    + ['revenue_rolling_4', 'revenue_rolling_4_std',
       'revenue_rolling_8', 'order_count_rolling_4', 'order_count_rolling_8']
)
print('✅ Import xong.')

✅ Import xong.


## 2. Load dữ liệu & tách Train/Val

In [3]:
df = load_revenue_data().sort_values(['year', 'week_of_year']).reset_index(drop=True)
df['time_label'] = df['year'].astype(str) + '-W' + df['week_of_year'].astype(str).str.zfill(2)

print(f'✅ {len(df)} tuần | {df["time_label"].iloc[0]} → {df["time_label"].iloc[-1]}')
df.tail(3)

✅ 76 tuần | 2016-W40 → 2018-W21


,year,week_of_year,month,weekly_revenue,order_count,weekend_order_ratio,time_label
73,2018,19,5,334320.93,1943,0.208441,2018-W19
74,2018,20,5,295686.95,1805,0.163435,2018-W20
75,2018,21,5,97457.71,609,0.000000,2018-W21


## 3. Load model + scaler từ MLflow Registry

In [4]:
client = MlflowClient()

def get_latest_run_id(model_name):
    versions = client.search_model_versions(f"name='{model_name}'")
    latest   = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
    print(f'  {model_name}: version={latest.version} | run_id={latest.run_id[:8]}...')
    return latest.version, latest.run_id

lgb_ver,  lgb_run_id  = get_latest_run_id('revenue_lightgbm')
lstm_ver, lstm_run_id = get_latest_run_id('revenue_lstm')

lgb_model  = mlflow.lightgbm.load_model(f'models:/revenue_lightgbm/{lgb_ver}')
lstm_model = mlflow.pytorch.load_model(f'models:/revenue_lstm/{lstm_ver}')
lstm_model.eval()

lgb_scaler    = mlflow.sklearn.load_model(f'runs:/{lgb_run_id}/scaler')
feat_scaler   = mlflow.sklearn.load_model(f'runs:/{lstm_run_id}/feat_scaler')
target_scaler = mlflow.sklearn.load_model(f'runs:/{lstm_run_id}/target_scaler')

print('✅ Load model + scaler thành công!')

  revenue_lightgbm: version=1 | run_id=0ae3e3a5...
  revenue_lstm: version=1 | run_id=4e375cdd...


✅ Load model + scaler thành công!


## 4. Inference LightGBM

In [5]:
df_lag   = build_lag_features(df.copy())
split_lg = int(len(df_lag) * TRAIN_RATIO)

X_all = df_lag[LGB_FEATURES].values
y_all = df_lag[TARGET_COL].values

X_val_s   = lgb_scaler.transform(X_all[split_lg:])
lgb_preds = np.maximum(np.expm1(lgb_model.predict(X_val_s)), 0)
y_val_lgb = y_all[split_lg:]

time_labels = (df_lag['year'].astype(str) + '-W' +
               df_lag['week_of_year'].astype(str).str.zfill(2)).tolist()

rmse_lgb = math.sqrt(mean_squared_error(y_val_lgb, lgb_preds))
mae_lgb  = mean_absolute_error(y_val_lgb, lgb_preds)
print(f'LightGBM → RMSE: {rmse_lgb:,.2f} | MAE: {mae_lgb:,.2f}')

LightGBM → RMSE: 54,507.40 | MAE: 41,047.97


c:\Users\vanthang\anaconda3\envs\dlh\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 5. Inference LSTM

In [6]:
from src.mlops.p1_revenue.train import add_calendar_features

df_seq = add_calendar_features(df.copy())
X_raw  = feat_scaler.transform(df_seq[FEATURE_COLS].values)
y_raw  = target_scaler.transform(df_seq[[TARGET_COL]].values)

X_seq, y_seq = [], []
for i in range(len(df_seq) - SEQ_LEN):
    X_seq.append(X_raw[i: i + SEQ_LEN])
    y_seq.append(y_raw[i + SEQ_LEN])
X_seq = np.array(X_seq, np.float32)
y_seq = np.array(y_seq, np.float32)

split_lstm  = int(len(X_seq) * TRAIN_RATIO)
X_val_lstm  = torch.tensor(X_seq[split_lstm:])
y_val_lstm  = y_seq[split_lstm:]

device = next(lstm_model.parameters()).device
with torch.no_grad():
    preds_norm = lstm_model(X_val_lstm.to(device)).cpu().numpy()

lstm_preds      = target_scaler.inverse_transform(preds_norm).flatten()
y_val_lstm_orig = target_scaler.inverse_transform(y_val_lstm).flatten()

time_seq_all  = (df['year'].astype(str) + '-W' +
                 df['week_of_year'].astype(str).str.zfill(2)).tolist()
time_val_lstm = time_seq_all[SEQ_LEN + split_lstm:]

rmse_lstm = math.sqrt(mean_squared_error(y_val_lstm_orig, lstm_preds))
mae_lstm  = mean_absolute_error(y_val_lstm_orig, lstm_preds)
print(f'LSTM     → RMSE: {rmse_lstm:,.2f} | MAE: {mae_lstm:,.2f}')

LSTM     → RMSE: 75,975.17 | MAE: 70,802.90


## 6. Biểu đồ so sánh

In [7]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=time_labels, y=y_all,
    name='Thực tế', line=dict(color='#3B82F6', width=2)
))
fig.add_trace(go.Scatter(
    x=time_labels[split_lg:], y=lgb_preds,
    name=f'LightGBM (RMSE={rmse_lgb:,.0f})',
    line=dict(color='#EF4444', width=2, dash='dash')
))
fig.add_trace(go.Scatter(
    x=time_val_lstm, y=lstm_preds,
    name=f'LSTM (RMSE={rmse_lstm:,.0f})',
    line=dict(color='#F97316', width=2, dash='dot')
))

fig.add_shape(
    type='line',
    x0=time_labels[split_lg], x1=time_labels[split_lg],
    y0=0, y1=1, yref='paper',
    line=dict(color='gray', dash='dash', width=1)
)
fig.add_annotation(
    x=time_labels[split_lg], y=1, yref='paper',
    text='← Train | Validation →',
    showarrow=False, yanchor='bottom', font=dict(color='gray')
)

fig.update_layout(
    title=dict(text='P1 — Doanh thu Thực tế vs Dự báo (Backtesting)', x=0.5),
    xaxis_title='Tuần', yaxis_title='Doanh thu (BRL)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    template='plotly_white', height=500
)
fig.show()

pd.DataFrame({
    'Model':    ['LightGBM', 'LSTM'],
    'RMSE':     [round(rmse_lgb, 2), round(rmse_lstm, 2)],
    'MAE':      [round(mae_lgb, 2),  round(mae_lstm, 2)],
    'Champion': ['✅' if rmse_lgb < rmse_lstm else '', '✅' if rmse_lstm < rmse_lgb else '']
})

,Model,RMSE,MAE,Champion
0,LightGBM,54507.40,41047.97,✅
1,LSTM,75975.17,70802.90,
